# Complete PySpark Transformations and Actions Reference

Comprehensive guide covering ALL PySpark transformations and actions.

## 1. Setup and Data Creation

In [ ]:
# COPY THIS CODE INTO THE FIRST CELL OF THE JUPYTER NOTEBOOK
# This fixes the SPARK_HOME and PYTHONPATH issues

import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, sum as spark_sum, expr
import time

# Unset conflicting environment variables
os.environ.pop('SPARK_HOME', None)
os.environ.pop('PYTHONPATH', None)

# Java 11 compatibility options
java_opts = "--add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED"
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--driver-java-options "{java_opts}" pyspark-shell'

# Create Spark session
spark = SparkSession.builder \
    .appName("PartitioningBucketingDemo") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.extraJavaOptions", java_opts) \
    .config("spark.executor.extraJavaOptions", java_opts) \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print("Spark session created successfully!")


### Sample Data Creation

In [ ]:
from pyspark.sql.types import *
# Employee data
employees_data = [
    (1, "John", "Doe", 28, "Engineering", 75000, "New York", "2020-01-15"),
    (2, "Jane", "Smith", 34, "Marketing", 65000, "San Francisco", "2019-03-20"),
    (3, "Mike", "Johnson", 45, "Engineering", 95000, "New York", "2018-06-10"),
    (4, "Emily", "Davis", 29, "HR", 55000, "Chicago", "2021-02-01"),
    (5, "David", "Wilson", 38, "Engineering", 85000, "San Francisco", "2019-08-15"),
    (6, "Sarah", "Brown", 31, "Marketing", 70000, "New York", "2020-05-12"),
    (7, "Tom", "Miller", 42, "Finance", 80000, "Chicago", "2017-11-30"),
    (8, "Lisa", "Anderson", 26, "HR", 52000, "San Francisco", "2021-07-22"),
]

schema = StructType([
    StructField("id", IntegerType()), StructField("first_name", StringType()),
    StructField("last_name", StringType()), StructField("age", IntegerType()),
    StructField("department", StringType()), StructField("salary", IntegerType()),
    StructField("city", StringType()), StructField("hire_date", StringType())
])

df = spark.createDataFrame(employees_data, schema)
df.show()

## 2. TRANSFORMATIONS (Lazy Operations)

### 2.1 select() - Select columns

In [ ]:
df.select("first_name", "department", "salary").show()
df.select(col("first_name"), (col("salary") * 1.1).alias("new_salary")).show()

### 2.2 filter() / where() - Filter rows

In [ ]:
df.filter(col("salary") > 70000).show()
df.where((col("department") == "Engineering") & (col("age") < 40)).show()

### 2.3 withColumn() - Add/modify columns

In [ ]:
from pyspark.sql.functions import col, lit, concat

df.withColumn("bonus", col("salary") * 0.1).show()
df.withColumn("full_name", concat(col("first_name"), lit(" "), col("last_name"))).show()

### 2.4 withColumnRenamed() - Rename columns

In [ ]:
df.withColumnRenamed("first_name", "fname").show()

### 2.5 drop() - Remove columns

In [ ]:
df.drop("age", "hire_date").show()

### 2.6 distinct() - Unique rows

In [ ]:
df.select("department").distinct().show()

### 2.7 dropDuplicates() - Remove duplicates

In [ ]:
df.dropDuplicates(["department"]).show()

### 2.8 orderBy() / sort() - Sort data

In [ ]:
df.orderBy(col("salary").desc()).show()
df.sort("department", col("salary").desc()).show()

### 2.9 groupBy() - Group for aggregation

In [ ]:
df.groupBy("department").count().show()
df.groupBy("department").agg(avg("salary"), max("salary")).show()

### 2.10 join() - Join DataFrames

In [ ]:
dept_df = spark.createDataFrame([("Engineering", "Building A"), ("Marketing", "Building B")], ["department", "location"])
df.join(dept_df, "department", "inner").show()

### 2.11 union() - Combine DataFrames

In [ ]:
df1 = df.filter(col("department") == "Engineering")
df2 = df.filter(col("department") == "HR")
df1.union(df2).show()

### 2.12 limit() - Limit rows

In [ ]:
df.limit(3).show()

### 2.13 sample() - Random sample

In [ ]:
df.sample(False, 0.5).show()

### 2.14 repartition() - Change partitions

In [ ]:
df_repart = df.repartition(4)
print(f"Partitions: {df_repart.rdd.getNumPartitions()}")

### 2.15 coalesce() - Reduce partitions

In [ ]:
df_coal = df.repartition(8).coalesce(2)
print(f"Partitions: {df_coal.rdd.getNumPartitions()}")

### 2.16 cache() / persist() - Cache data

In [ ]:
df_cached = df.cache()
print("DataFrame cached")

### 2.17 fillna() - Fill nulls

In [ ]:
df.fillna({"age": 0, "salary": 50000}).show()

### 2.18 dropna() - Drop nulls

In [ ]:
df.dropna(subset=["age"]).show()

### 2.19 replace() - Replace values

In [ ]:
df.replace("New York", "NYC", subset=["city"]).show()

### 2.20 when() / otherwise() - Conditional logic

In [ ]:
df.withColumn("level", when(col("salary") >= 80000, "High").otherwise("Low")).show()

### 2.21 selectExpr() - SQL expressions

In [ ]:
df.selectExpr("first_name", "salary * 1.1 as new_salary").show()

### 2.22 alias() - Rename DataFrame

In [ ]:
df.alias("employees").select("employees.first_name").show()

### 2.23 crossJoin() - Cartesian product

In [ ]:
df.select("first_name").limit(2).crossJoin(df.select("department").limit(2)).show()

### 2.24 Window Functions - Analytics

In [ ]:
w = Window.partitionBy("department").orderBy(col("salary").desc())
df.withColumn("rank", row_number().over(w)).show()

### 2.25 pivot() - Pivot table

In [ ]:
df.groupBy("city").pivot("department").count().show()

### 2.26 explode() - Expand arrays

In [ ]:
arr_df = spark.createDataFrame([(1, ["a", "b"]), (2, ["c"])], ["id", "vals"])
arr_df.select("id", explode("vals").alias("val")).show()

### 2.27 collect_list() / collect_set() - Aggregate to array

In [ ]:
df.groupBy("department").agg(collect_list("first_name")).show(truncate=False)

### 2.28 cast() - Type conversion

In [ ]:
df.select(col("salary").cast("double")).printSchema()

### 2.29 toDF() - Rename all columns

In [ ]:
df.toDF("emp_id", "fname", "lname", "emp_age", "dept", "sal", "loc", "date").show()

## 3. ACTIONS (Eager Operations)

### 3.1 show() - Display data

In [ ]:
df.show()
df.show(5, truncate=False)

### 3.2 count() - Count rows

In [ ]:
print(f"Total rows: {df.count()}")

### 3.3 collect() - Get all rows

In [ ]:
rows = df.limit(3).collect()
for row in rows:
    print(row)

### 3.4 take() - Take N rows

In [ ]:
print(df.take(2))

### 3.5 first() - First row

In [ ]:
print(df.first())

### 3.6 head() - Head rows

In [ ]:
print(df.head(3))

### 3.7 tail() - Last rows

In [ ]:
print(df.tail(2))

### 3.8 describe() - Statistics

In [ ]:
df.describe().show()

### 3.9 summary() - Extended stats

In [ ]:
df.summary().show()

### 3.10 printSchema() - Schema

In [ ]:
df.printSchema()

### 3.11 columns - Column names

In [ ]:
print(df.columns)

### 3.12 dtypes - Data types

In [ ]:
print(df.dtypes)

### 3.13 foreach() - Apply function

In [ ]:
df.limit(3).foreach(lambda row: print(row.first_name))

### 3.14 toPandas() - Convert to Pandas

In [ ]:
pdf = df.limit(5).toPandas()
print(type(pdf))

### 3.15 write - Save data

In [ ]:
df.write.mode("overwrite").parquet("/tmp/pyspark_output")
print("Data written")

### 3.16 createTempView() - SQL view

In [ ]:
df.createOrReplaceTempView("employees")
spark.sql("SELECT department, AVG(salary) FROM employees GROUP BY department").show()

## 4. Advanced Operations

### 4.1 UDF - User Defined Function

In [ ]:
from pyspark.sql.functions import udf
categorize = udf(lambda x: "High" if x > 70000 else "Low", StringType())
df.withColumn("category", categorize(col("salary"))).show()

### 4.2 Broadcast Join

In [ ]:
from pyspark.sql.functions import broadcast
small_df = spark.createDataFrame([("Engineering", "A")], ["department", "code"])
df.join(broadcast(small_df), "department").show()

### 4.3 String Functions

In [ ]:
df.select(upper("first_name"), lower("last_name"), length("first_name")).show()

### 4.4 Date Functions

In [ ]:
df.select(to_date("hire_date"), year(to_date("hire_date"))).show()

### 4.5 Math Functions

In [ ]:
df.select(round(col("salary")/12, 2).alias("monthly")).show()

### Summary
#### Date : 05-Dec-2025 

This notebook almost covers all major PySpark transformations and actions . I will keep on adding more ...